# Sähkön hinta: juurisyyanalyysi

Miksi sähkölasku voi nousta, vaikka keskihinta laskee?

Vertaillaan yksinkertaista ja kulutuksella painotettua keskihintaa
Suomen spot-hinnoilla ja kansallisella kulutusdatalla.

**Lähteet**
- Spot-hinnat: sahkotin.fi (Pakastin Oy), data Nord Pool
- Kulutus: Fingrid avoin data, dataset 124 (CC 4.0 BY)

## 1. Setup

In [1]:
import os
import time
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

avain = os.getenv("FINGRID_API_KEY")
print("avain ladattu:", avain is not None, "| pituus:", len(avain) if avain else 0)

avain ladattu: True | pituus: 32


## 2. Apufunktiot

Hinnat haetaan ilman ALV:ta, koska yritys vähentää sen ja
ALV-kanta muuttui 9/2024 (24 % -> 25,5 %). ALV:llisilla hinnoilla
osa vuosimuutoksesta olisi verotusta, ei markkinaa.

In [2]:
def tarkista(df, nimi, odotettu_rivit, sarake):
    """Datan laatutarkistus: rivimäärä, puuttuvat, duplikaatit, jakauma."""
    print(f"--- {nimi} ---")
    print("rivejä:", len(df), "| odotettu:", odotettu_rivit)
    print("puuttuvia:", df.isna().sum().sum())
    print("duplikaatteja:", df["ts"].duplicated().sum())
    print("aikaväli:", df["ts"].min(), "->", df["ts"].max())
    print(df[sarake].describe())

In [3]:
def hae_hinnat(start, end):
    """Spot-hinnat sahkotin.fi:stä, snt/kWh ilman ALV:ta, UTC."""
    r = requests.get("https://sahkotin.fi/prices", params={
        "start": start,
        "end": end,
        "fix": ""          # €/MWh -> snt/kWh; ei vat-parametria
    }, timeout=30)
    r.raise_for_status()

    df = pd.DataFrame(r.json()["prices"])
    df["ts"] = pd.to_datetime(df["date"], utc=True)
    return (df[["ts", "value"]]
            .rename(columns={"value": "price_snt_kwh"})
            .sort_values("ts")
            .reset_index(drop=True))

In [4]:
def hae_kulutus(start, end, dataset_id=124):
    """Suomen sähkönkulutus Fingridiltä, MWh/h, 15 min resoluutio, UTC."""
    rows, page = [], 1
    while True:
        r = requests.get(
            f"https://data.fingrid.fi/api/datasets/{dataset_id}/data",
            headers={"x-api-key": os.getenv("FINGRID_API_KEY")},
            params={"startTime": start, "endTime": end,
                    "pageSize": 20000, "page": page,
                    "sortBy": "startTime", "sortOrder": "asc"},
            timeout=60)
        r.raise_for_status()
        payload = r.json()
        rows.extend(payload["data"])

        p = payload["pagination"]
        if p["currentPage"] >= p["lastPage"]:
            print(f"  Fingrid: {p['total']} riviä, {p['lastPage']} sivua")
            break
        page += 1
        time.sleep(7)      # kiintiö 10 kutsua/min

    df = pd.DataFrame(rows)
    df["ts"] = pd.to_datetime(df["startTime"], utc=True)
    return (df[["ts", "value"]]
            .rename(columns={"value": "consumption_mwh"})
            .sort_values("ts")
            .reset_index(drop=True))

## 3. Datan haku ja tarkistus

Testataan ensin yhdellä kuukaudella (tammikuu 2024 = 744 tuntia)
ennen koko aikavälin hakua.

In [5]:
prices = hae_hinnat("2024-01-01T00:00:00.000Z", "2024-02-01T00:00:00.000Z")
tarkista(prices, "Hinnat 1/2024", 744, "price_snt_kwh")

--- Hinnat 1/2024 ---
rivejä: 744 | odotettu: 744
puuttuvia: 0
duplikaatteja: 0
aikaväli: 2024-01-01 00:00:00+00:00 -> 2024-01-31 23:00:00+00:00
count    744.000000
mean      10.616461
std       18.693882
min       -0.207000
25%        3.156750
50%        7.814500
75%       11.009750
max      189.600000
Name: price_snt_kwh, dtype: float64


In [6]:
raaka = hae_kulutus("2024-01-01T00:00:00Z", "2024-02-01T00:00:00Z")

print("raakarivejä:", len(raaka), "| odotettu 15 min:", 744 * 4)
print("aikaväli:", raaka["ts"].min(), "->", raaka["ts"].max())

  Fingrid: 2976 riviä, 1 sivua
raakarivejä: 2976 | odotettu 15 min: 2976
aikaväli: 2024-01-01 00:00:00+00:00 -> 2024-01-31 23:45:00+00:00


### Kulutuksen aggregointi tunneiksi

Kulutus on varttitasolla (13.6.2023 alkaen), hinnat tuntitasolla
1.10.2025 asti. Yhdenmukaistetaan tuntitasolle.

Aggregointi tehdään **keskiarvona, ei summana**: MWh/h on teho, ei
energia. Neljän vartin keskiarvo antaa tunnin keskitehon, joka
MWh/h-yksikössä vastaa tunnin energiaa. Summaus antaisi
nelinkertaisen kulutuksen.

In [7]:
consumption = (raaka.set_index("ts")
                    .resample("1h")["consumption_mwh"]
                    .mean()
                    .reset_index())

tarkista(consumption, "Kulutus 1/2024", 744, "consumption_mwh")

--- Kulutus 1/2024 ---
rivejä: 744 | odotettu: 744
puuttuvia: 0
duplikaatteja: 0
aikaväli: 2024-01-01 00:00:00+00:00 -> 2024-01-31 23:00:00+00:00
count      744.000000
mean     12426.091509
std       1128.159631
min       9530.455000
25%      11615.812500
50%      12430.762500
75%      13295.925000
max      14995.850000
Name: consumption_mwh, dtype: float64
